# Grover search

Unstructured search over $N=2^n$ addresses costs $O(\sqrt{N})$ oracle
queries. One Grover iterate is

$$
G = (2|s\rangle\langle s|-I)\,O_f
$$

where $|s\rangle$ is the uniform superposition and $O_f$ phase-flips
the marked item.

Here $n=3$, $N=8$, the marked state is $|101\rangle$, and the optimal
iteration count is $\lfloor \pi/4\sqrt{8}\rfloor = 2$.

Oracle and diffuser are written out as gates. This notebook does not
import the Shor example.

In [ ]:
import qiskit as qk
import qiskit_aer as qka


class std:
    import math


N = 3
MARKED = "101"
print("optimal k =", int(std.math.floor(std.math.pi / 4 * std.math.sqrt(2 ** N))))

## Phase oracle

Flip every qubit that should be $0$ in the marked string, apply a
multi-controlled $Z$ (H–MCX–H on the last qubit), then undo the flips.

In [ ]:
def oracle(marked):
    qc = qk.QuantumCircuit(N, name="oracle")
    zeros = [i for i, bit in enumerate(reversed(marked)) if bit == "0"]
    for q in zeros:
        qc.x(q)
    qc.h(N - 1)
    qc.mcx(list(range(N - 1)), N - 1)
    qc.h(N - 1)
    for q in zeros:
        qc.x(q)
    return qc

print(oracle(MARKED).draw())

## Diffuser $2|s\rangle\langle s|-I$

In [ ]:
def diffuser(n):
    qc = qk.QuantumCircuit(n, name="diffuser")
    qc.h(range(n))
    qc.x(range(n))
    qc.h(n - 1)
    qc.mcx(list(range(n - 1)), n - 1)
    qc.h(n - 1)
    qc.x(range(n))
    qc.h(range(n))
    return qc

print(diffuser(N).draw())

## How the marked amplitude grows

In [ ]:
def marked_probability(k):
    qc = qk.QuantumCircuit(N)
    qc.h(range(N))
    for _ in range(k):
        qc.compose(oracle(MARKED), inplace=True)
        qc.compose(diffuser(N), inplace=True)
    return qk.quantum_info.Statevector.from_instruction(qc).probabilities_dict().get(MARKED, 0.0)

for k in range(0, 5):
    print(f"k={k}  p={marked_probability(k):.3f}")

## Sample the optimal circuit

In [ ]:
k = int(std.math.floor(std.math.pi / 4 * std.math.sqrt(2 ** N)))
qc = qk.QuantumCircuit(N, N)
qc.h(range(N))
for _ in range(k):
    qc.compose(oracle(MARKED), inplace=True)
    qc.compose(diffuser(N), inplace=True)
qc.measure(range(N), range(N))
print(qc.draw())

sim = qka.AerSimulator()
counts = sim.run(qk.transpile(qc, sim), shots=2000).result().get_counts()
print(counts)
qk.visualization.plot_histogram(counts)